# E-Commerce Recommendation & Sales Analysis
SQL + Python EDA + RFM + Market Basket Analysis + Recommendation System

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_csv('ecommerce_recommendation_sales_dataset.csv')
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
total_revenue = df['Revenue'].sum()
total_orders = df['Order_ID'].nunique()
total_customers = df['Customer_ID'].nunique()
aov = total_revenue / total_orders
print(total_revenue, total_orders, total_customers, aov)

In [ ]:
category_sales = df.groupby('Category')['Revenue'].sum().sort_values(ascending=False)
category_sales.plot(kind='bar', title='Category-wise Revenue')
plt.ylabel('Revenue')
plt.show()

In [ ]:
top_products = df.groupby('Product')['Revenue'].sum().sort_values(ascending=False).head(10)
top_products.plot(kind='bar', title='Top 10 Products by Revenue')
plt.ylabel('Revenue')
plt.show()

In [ ]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['YearMonth'] = df['Order_Date'].dt.to_period('M').astype(str)
monthly = df.groupby('YearMonth')['Revenue'].sum()
monthly.plot(kind='line', marker='o', title='Monthly Revenue Trend')
plt.xticks(rotation=45)
plt.ylabel('Revenue')
plt.show()

## RFM Customer Segmentation

In [ ]:
snapshot_date = df['Order_Date'].max() + pd.Timedelta(days=1)
rfm = df.groupby('Customer_ID').agg({
    'Order_Date': lambda x: (snapshot_date - x.max()).days,
    'Order_ID': 'nunique',
    'Revenue': 'sum'
}).reset_index()
rfm.columns = ['Customer_ID','Recency','Frequency','Monetary']
rfm.head()

In [ ]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4])
rfm['RFM_Score'] = rfm['R_Score'].astype(str)+rfm['F_Score'].astype(str)+rfm['M_Score'].astype(str)
rfm.head()

## Market Basket Analysis

In [ ]:
basket = df.groupby(['Order_ID','Product'])['Quantity'].sum().unstack().fillna(0)
basket = basket.map(lambda x: 1 if x > 0 else 0)
basket.head()

In [ ]:
frequent_items = apriori(basket.astype(bool), min_support=0.02, use_colnames=True)
rules = association_rules(frequent_items, metric='lift', min_threshold=1)
rules = rules.sort_values(['lift','confidence'], ascending=False)
rules[['antecedents','consequents','support','confidence','lift']].head(20)

In [ ]:
def recommend_products(product, top_n=5):
    rec = rules[rules['antecedents'].apply(lambda x: product in list(x))].copy()
    rec = rec.sort_values(['lift','confidence'], ascending=False)
    rec['Recommended_Product'] = rec['consequents'].apply(lambda x: ', '.join(list(x)))
    return rec[['Recommended_Product','support','confidence','lift']].head(top_n)

recommend_products('Laptop')

In [ ]:
rules_out = rules.copy()
rules_out['antecedents'] = rules_out['antecedents'].apply(lambda x: ', '.join(list(x)))
rules_out['consequents'] = rules_out['consequents'].apply(lambda x: ', '.join(list(x)))
rules_out[['antecedents','consequents','support','confidence','lift']].to_csv('product_affinity_rules.csv', index=False)
rfm.to_csv('customer_rfm_segments.csv', index=False)